<a href="https://colab.research.google.com/github/sanmaykant/sec/blob/main/Notebooks/topic_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bertopic umap-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/sec-risk-analyzer"

EMBED_PATH = os.path.join(BASE_PATH, "data/embeddings")
MODEL_PATH = os.path.join(BASE_PATH, "models")
ARTIFACT_PATH = os.path.join(BASE_PATH, "artifacts/topic_data")

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(ARTIFACT_PATH, exist_ok=True)

print("Paths ready!")

Paths ready!


In [ ]:
import numpy as np
import json

embeddings = np.load(os.path.join(EMBED_PATH, "all_embeddings.npy"))

with open(os.path.join(EMBED_PATH, "metadata.json"), "r") as f:
    metadata = json.load(f)

with open(os.path.join(BASE_PATH, "data/processed/chunks/all_chunks.json"), "r") as f:
    chunk_data = json.load(f)

docs = chunk_data["chunks"]

print("Docs:", len(docs))
print("Embeddings:", embeddings.shape)

Docs: 1649
Embeddings: (1649, 768)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
)

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

2026-04-13 15:21:43,867 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-13 15:21:53,975 - BERTopic - Dimensionality - Completed ✓
2026-04-13 15:21:53,977 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 15:21:54,300 - BERTopic - Cluster - Completed ✓
2026-04-13 15:21:54,307 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-13 15:21:54,689 - BERTopic - Representation - Completed ✓


In [ ]:
topics_full = topics.copy()

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,108,-1_company_investigations_regulations_companys,"[company, investigations, regulations, company...",[The People’s Republic of China (“PRC”) and In...
1,0,71,0_tax_taxes_tax rate_effective tax,"[tax, taxes, tax rate, effective tax, rate, ef...",[The tax regimes we are subject to or operate ...
2,1,67,1_acquisitions_strategic_transactions_acquire,"[acquisitions, strategic, transactions, acquir...","[We provide physical, e-commerce, and omnichan..."
3,2,52,2_laws_regulations_protection_data,"[laws, regulations, protection, data, content,...",[We have been subject to other significant leg...
4,3,52,3_ads_advertising_marketers_ability,"[ads, advertising, marketers, ability, effecti...",[•our ability to attract and retain marketers ...
5,4,51,4_payments_payment_card_money,"[payments, payment, card, money, subject, card...",[We accept payments using a variety of methods...
6,5,47,5_mobile_developers_signals_products,"[mobile, developers, signals, products, mobile...",[The substantial majority of our revenue is ge...
7,6,40,6_stock_changes_market_estimates,"[stock, changes, market, estimates, securities...",[We Have a Rapidly Evolving Business Model and...
8,7,39,7_products_users_engagement_products services,"[products, users, engagement, products service...",[•user behavior on any of our products changes...
9,8,38,8_extent_costs_ability_costs related,"[extent, costs, ability, costs related, relate...",[our ability to maintain gross margins and ope...


In [ ]:
for topic_id in topic_info["Topic"].head(10):
    print(f"\nTopic {topic_id}:")
    print(topic_model.get_topic(topic_id))


Topic -1:
[('company', np.float64(0.014089698353267914)), ('investigations', np.float64(0.011566945472617792)), ('regulations', np.float64(0.011099380021472784)), ('companys', np.float64(0.010728981355207277)), ('governance', np.float64(0.010289194756864567)), ('intellectual', np.float64(0.01025716184488836)), ('intellectual property', np.float64(0.01025716184488836)), ('government', np.float64(0.010195966610120701)), ('property', np.float64(0.010007888682673211)), ('prc', np.float64(0.009675584692473316))]

Topic 0:
[('tax', np.float64(0.08415347493202222)), ('taxes', np.float64(0.035100213895617176)), ('tax rate', np.float64(0.02680084241808706)), ('effective tax', np.float64(0.0250752718011818)), ('rate', np.float64(0.02285388725255445)), ('effective', np.float64(0.021808513547596806)), ('changes', np.float64(0.02087827470195172)), ('tax laws', np.float64(0.01945371822584187)), ('obligations', np.float64(0.018759244702705135)), ('income', np.float64(0.01863971466535115))]

Topic 1:

In [ ]:
topic_info_clean = topic_info[topic_info["Topic"] != -1]
topic_info_clean.head()

,Topic,Count,Name,Representation,Representative_Docs
1,0,71,0_tax_taxes_tax rate_effective tax,"[tax, taxes, tax rate, effective tax, rate, ef...",[The tax regimes we are subject to or operate ...
2,1,67,1_acquisitions_strategic_transactions_acquire,"[acquisitions, strategic, transactions, acquir...","[We provide physical, e-commerce, and omnichan..."
3,2,52,2_laws_regulations_protection_data,"[laws, regulations, protection, data, content,...",[We have been subject to other significant leg...
4,3,52,3_ads_advertising_marketers_ability,"[ads, advertising, marketers, ability, effecti...",[•our ability to attract and retain marketers ...
5,4,51,4_payments_payment_card_money,"[payments, payment, card, money, subject, card...",[We accept payments using a variety of methods...


In [ ]:
enriched_data = []

for i in range(len(docs)):
    enriched_data.append({
        "chunk": docs[i],
        "topic": int(topics[i]),
        "year": metadata[i]["year"],
        "ticker": metadata[i]["ticker"]
    })

print("Enriched dataset ready!")

Enriched dataset ready!


In [ ]:
topic_model.save(os.path.join(MODEL_PATH, "bertopic_model"))
print("Model saved!")

2026-04-13 15:19:04,914 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Model saved!


In [ ]:
import json

with open(os.path.join(ARTIFACT_PATH, "enriched_chunks.json"), "w") as f:
    json.dump(enriched_data, f)

print("Enriched data saved!")

Enriched data saved!


In [ ]:
import numpy as np
topic_embeddings = topic_model.topic_embeddings_
np.save(os.path.join(ARTIFACT_PATH, "topic_embeddings.npy"), topic_embeddings)

print("Topic embeddings saved!")

Topic embeddings saved!


In [ ]:
topic_info.to_csv(os.path.join(ARTIFACT_PATH, "topic_info.csv"), index=False)
print("Topic info saved!")

Topic info saved!


In [ ]:
topic_model_reduced = topic_model.reduce_topics(docs, nr_topics=20)
topics_reduced = topic_model.topics_

2026-04-13 15:22:25,570 - BERTopic - Topic reduction - Reducing number of topics
2026-04-13 15:22:25,618 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-13 15:22:26,727 - BERTopic - Representation - Completed ✓
2026-04-13 15:22:26,733 - BERTopic - Topic reduction - Reduced number of topics from 64 to 20


In [ ]:
topic_model.save(os.path.join(MODEL_PATH, "bertopic_model_reduced"))
with open(os.path.join(ARTIFACT_PATH, "topics_reduced.json"), "w") as f:
    json.dump(topics_reduced, f)

print("Reduced model saved!")

2026-04-13 15:22:35,806 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Reduced model saved!


In [ ]:
# Analysis

In [ ]:
from collections import defaultdict
year_topics = defaultdict(set)

for item in enriched_data:
    if item["topic"] != -1:
        year_topics[item["year"]].add(item["topic"])

for year in sorted(year_topics):
    print(year, "→", len(year_topics[year]), "topics")

2020 → 65 topics
2021 → 65 topics
2022 → 65 topics
2023 → 65 topics
2024 → 65 topics


In [ ]:
company_topics = defaultdict(set)

for item in enriched_data:
    if item["topic"] != -1:
        company_topics[item["ticker"]].add(item["topic"])

for comp in company_topics:
    print(comp, "→", len(company_topics[comp]), "topics")

AAPL → 34 topics
META → 48 topics
AMZN → 24 topics


In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_hierarchy()